In [1]:
import torch.nn as nn
import torch
loss_func=nn.BCELoss()
loss=loss_func(torch.tensor([0.9]),torch.tensor([1.0]))
l2_lambda = 0.001
conv_layer=nn.Conv2d(in_channels=3,out_channels=5,kernel_size=5)
l2_penalty=l2_lambda*sum([(p**2).sum() for p in conv_layer.parameters()])

loss_with_penalty=loss+l2_penalty
linear_layer=nn.Linear(10,16)
l2_penalty = l2_lambda * sum([(p**2).sum() for p in linear_layer.parameters()] )
loss_with_penalty = loss + l2_penalty

In [3]:
import torchvision
from torchvision import transforms
image_path='./'
transform=transforms.Compose([transforms.ToTensor()])
mnist_dataset=torchvision.datasets.MNIST(root=image_path,train=True,transform=transform,download=True)
from torch.utils.data import Subset
mnist_valid_dataset=Subset(mnist_dataset,torch.arange(10000))
mnist_train_dataset=Subset(mnist_dataset,torch.arange(10000,len(mnist_dataset)))
mnist_test_dataset=torchvision.datasets.MNIST(root=image_path,train=False,transform=transform,download=False)


In [11]:
from torch.utils.data import DataLoader
batch_size=64
torch.manual_seed(1)
train_dl=DataLoader(mnist_train_dataset,batch_size,shuffle=True)
valid_dl=DataLoader(mnist_valid_dataset,batch_size,shuffle=False)


In [4]:
model=nn.Sequential()
model.add_module('conv1',nn.Conv2d(in_channels=1,out_channels=32,kernel_size=5,padding=2))
model.add_module('relu',nn.ReLU())
model.add_module('pool1',nn.MaxPool2d(kernel_size=2))
model.add_module('conv2',nn.Conv2d(in_channels=32,out_channels=64,kernel_size=5,padding=2))
model.add_module('relu2',nn.ReLU())
model.add_module('pool2',nn.MaxPool2d(kernel_size=2))

In [5]:
x=torch.ones((4,1,28,28))
model(x).shape


torch.Size([4, 64, 7, 7])

In [6]:
model.add_module('flatten',nn.Flatten())
x=torch.ones((4,1,28,28))
model(x).shape

torch.Size([4, 3136])

In [7]:
model.add_module('fc1',nn.Linear(3136,1024))
model.add_module('relu3',nn.ReLU())
model.add_module('dropout',nn.Dropout(p=0.5))
model.add_module('fc2',nn.Linear(1024,10))

In [8]:
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)


In [9]:
def train(model, num_epochs, train_dl, valid_dl):
 loss_hist_train = [0] * num_epochs
 accuracy_hist_train = [0] * num_epochs
 loss_hist_valid = [0] * num_epochs
 accuracy_hist_valid = [0] * num_epochs
 for epoch in range(num_epochs):
  model.train()
  for x_batch, y_batch in train_dl:
   pred = model(x_batch)
   loss = loss_fn(pred, y_batch)
   loss.backward()
   optimizer.step()
   optimizer.zero_grad()
   loss_hist_train[epoch] += loss.item()*y_batch.size(0)
   is_correct = (torch.argmax(pred, dim=1) == y_batch).float()
   accuracy_hist_train[epoch] += is_correct.sum()
  loss_hist_train[epoch] /= len(train_dl.dataset)
  accuracy_hist_train[epoch] /= len(train_dl.dataset)
  model.eval()

  with torch.no_grad():
   for x_batch,y_batch in valid_dl:
    pred=model(x_batch)
    loss=loss_fn(pred,y_batch)
    loss_hist_valid[epoch]+=loss.item()*y_batch.size(0)
    is_correct=(torch.argmax(pred,dim=1)==y_batch).float()
    accuracy_hist_valid[epoch]+=is_correct.sum()
    loss_hist_valid[epoch] /= len(valid_dl.dataset)
    accuracy_hist_valid[epoch] /= len(valid_dl.dataset)
    print(f'Epoch {epoch+1} accuracy: '
f'{accuracy_hist_train[epoch]:.4f} val_accuracy: '
f'{accuracy_hist_valid[epoch]:.4f}')
    return loss_hist_train, loss_hist_valid, accuracy_hist_train, accuracy_hist_valid



In [13]:
import os
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
os.environ["CUDA_VISIBLE_DEVICES"] = ""


CUDA_VISIBLE_DEVICES: None


In [14]:
torch.manual_seed(1)
num_epochs=20
hist = train(model, num_epochs, train_dl, valid_dl)

Epoch 1 accuracy: 0.9898 val_accuracy: 0.0064


celeba

In [15]:
image_path='./'
celeba_train_dataset=torchvision.datasets.CelebA(image_path,split='train',target_type='attr',download=False)
celeba_valid_dataset=torchvision.datasets.CelebA(image_path,split='valid',target_type='attr',download=False)
celeba_test_dataset=torchvision.datasets.CelebA(image_path,split='test',target_type='attr',download=False)

print('Train set:', len(celeba_train_dataset))
print('Validation set:', len(celeba_valid_dataset))
print('Test set:', len(celeba_test_dataset))

Train set: 162770
Validation set: 19867
Test set: 19962


In [17]:
get_smile=lambda attr:attr[31]
transform_train=transforms.Compose([transforms.RandomCrop([178,178]),transforms.RandomHorizontalFlip(),transforms.Resize([64,64]),transforms.ToTensor(),])

In [30]:
transform_valid=transforms.Compose([transforms.CenterCrop([178,178]),transforms.Resize([64,64]),transforms.ToTensor(),])

In [31]:
model=nn.Sequential()
model.add_module("conv1",nn.Conv2d(in_channels=3,out_channels=32,kernel_size=3,padding=1))
model.add_module("relu1",nn.ReLU())
model.add_module("pool1",nn.MaxPool2d(kernel_size=2))
model.add_module("dropout1",nn.Dropout(p=0.5))
model.add_module('conv2',nn.Conv2d(in_channels=32,out_channels=64,kernel_size=3,padding=1))
model.add_module('relu2',nn.ReLU())
model.add_module('pool2',nn.MaxPool2d(kernel_size=2))
model.add_module('dropout2',nn.Dropout(p=0.5))
model.add_module('conv3',nn.Conv2d(in_channels=64,out_channels=128,kernel_size=3,padding=1))
model.add_module('relu3',nn.ReLU())
model.add_module('pool3',nn.MaxPool2d(kernel_size=2))
model.add_module('conv4',nn.Conv2d(in_channels=128,out_channels=256,kernel_size=3,padding=1))
model.add_module('relu4',nn.ReLU())

In [32]:
x=torch.ones((4,3,64,64))
model(x).shape

torch.Size([4, 256, 8, 8])

In [33]:
model.add_module('pool4',nn.AvgPool2d(kernel_size=8))
model.add_module('flatten',nn.Flatten())
x=torch.ones((4,3,64,64))
model(x).shape


torch.Size([4, 256])

In [34]:
model.add_module('linear',nn.Linear(256,1))
model.add_module('sigmoid',nn.Sigmoid())
x=torch.ones((4,3,64,64))
model(x).shape

torch.Size([4, 1])

In [35]:
model

Sequential(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu1): ReLU()
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout1): Dropout(p=0.5, inplace=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu2): ReLU()
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout2): Dropout(p=0.5, inplace=False)
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu3): ReLU()
  (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv4): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu4): ReLU()
  (pool4): AvgPool2d(kernel_size=8, stride=8, padding=0)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear): Linear(in_features=256, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [36]:
loss_fn=nn.BCELoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)

In [37]:
def train(model,num_epochs,train_dl,valid_dl):
    loss_hist_train=[0]*num_epochs
    accuracy_hist_train=[0]*num_epochs
    loss_hist_valid=[0]*num_epochs
    accuracy_hist_valid=[0]*num_epochs
    for epoch in range(num_epochs):
        model.train()
        for x_batch,y_batch in train_dl:
            pred=model(x_batch)[:,0]
            loss=loss_fn(pred,y_batch.float())
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            loss_hist_train[epoch]+=loss.item()*y_batch.size(0)
            is_correct=((pred>=0.5).float()==y_batch).float()
            accuracy_hist_train[epoch]+=is_correct.sum()
        loss_hist_train[epoch]/=len(train_dl.dataset)
        accuracy_hist_train[epoch]/=len(train_dl.dataset)

        model.eval()
        with torch.no_grad():
            for x_batch,y_batch in valid_dl:
                pred=model(x_batch)[:,0]
                loss = loss_fn(pred, y_batch.float())  
                loss_hist_valid[epoch] += loss.item() * y_batch.size(0)
                is_correct =((pred>=0.5).float() == y_batch).float()
                accuracy_hist_valid[epoch] += is_correct.sum()
        loss_hist_valid[epoch] /= len(valid_dl.dataset)
        accuracy_hist_valid[epoch] /= len(valid_dl.dataset)
        print(f'Epoch {epoch+1} accuracy: '
              f'{accuracy_hist_train[epoch]:.4f} val_accuracy: '
              f'{accuracy_hist_valid[epoch]:.4f}')
        
        return loss_hist_train, loss_hist_valid,accuracy_hist_train, accuracy_hist_valid



In [38]:
torch.manual_seed(1)
num_epochs=30
hist=train(model,num_epochs,train_dl,valid_dl)

RuntimeError: Given groups=1, weight of size [32, 3, 3, 3], expected input[64, 1, 28, 28] to have 3 channels, but got 1 channels instead